# LightAutoML (LAMA) Baselines

В данном ноутбуке представлены базовые модели на основе библиотеки LightAutoML (LAMA)

**Цель**: построить baseline-модели для задачи регрессии (предсказание страховых выплат) и сравнить различные конфигурации LAMA.

Log MAE — средняя абсолютная ошибка на логарифмированном таргете (официальная метрика соревнования Allstate Claims Severity).


In [1]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from src.data.loader import DataLoader
from src.features.eda_insights import load_eda_insights
from src.models.baseline import LAMABaseline
from src.utils.logging_config import setup_logging, get_logger
from src.utils.metrics import compute_log_mae, compute_metrics

warnings.filterwarnings('ignore')
setup_logging()
logger = get_logger(__name__)

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
%matplotlib inline

In [2]:
loader = DataLoader()
train_df = loader.load_train()
test_df = loader.load_test()

2025-12-20 21:10:28,473 - src.data.loader - INFO - Loaded 188,318 training samples
2025-12-20 21:10:28,858 - src.data.loader - INFO - Loaded 125,546 test samples


## Конфигурация 1 (default)

Базовая конфигурация LAMA использует простой одноуровневый ансамбль с блендингом:

**Алгоритмы:**
- `lgb` — LightGBM (градиентный бустинг)
- `linear_l2` — Ridge-регрессия (линейная модель с L2-регуляризацией)

**Особенности:**
- 5-fold кросс-валидация
- Финальное предсказание: взвешенная комбинация моделей (blending)
- Используются алгоритмы с дефолтными гиперпараметрами (без тюнинга)
- Веса блендинга оптимизируются автоматически на основе CV-скоров

In [3]:
eda_insights = load_eda_insights()

lama_default = LAMABaseline(
    config_name="default",
    timeout=600,
    n_threads=4,
    n_folds=5,
    random_state=42,
    eda_insights=eda_insights,
)

lama_default.fit(train_df, target_col="loss", drop_cols=["id"])

y_true = train_df["loss"].values
metrics_default = lama_default.evaluate(y_true, return_all_metrics=True)

print(f"Log MAE: {metrics_default['log_mae']:.4f}")


2025-12-20 21:10:28,862 - src.models.baseline - INFO - Fitting LAMA baseline (default), shape: (188318, 132)
[21:10:28] Stdout logging level is INFO.
[21:10:28] Copying TaskTimer may affect the parent PipelineTimer, so copy will create new unlimited TaskTimer
[21:10:28] Task: reg

[21:10:28] Start automl preset with listed constraints:
[21:10:28] - time: 600.00 seconds
[21:10:28] - CPU: 4 cores
[21:10:28] - memory: 16 GB

[21:10:28] Train data shape: (188318, 132)

[21:10:34] Layer 1 train process start. Time left 594.24 secs
[21:10:38] Start fitting Lvl_0_Pipe_0_Mod_0_LinearL2 ...
[21:11:07] Fitting Lvl_0_Pipe_0_Mod_0_LinearL2 finished. score = -1302.0093579111804
[21:11:07] Lvl_0_Pipe_0_Mod_0_LinearL2 fitting and predicting completed
[21:11:07] Time left 561.54 secs

[21:11:10] Selector_LightGBM fitting and predicting completed
[21:11:14] Start fitting Lvl_0_Pipe_1_Mod_0_LightGBM ...
[21:11:34] Fitting Lvl_0_Pipe_1_Mod_0_LightGBM finished. score = -1909.5202468069765
[21:11:34] Lvl_0

## Конфигурация 2 (tuned)

Улучшенная конфигурация LAMA использует многоуровневую архитектуру

**Архитектура:**
- **Уровень 1:** `linear_l2` + `lgb` — базовые модели
- **Уровень 2:** `lgb_tuned` — LightGBM с оптимизацией гиперпараметров через Optuna

**Особенности:**
- 5-fold кросс-валидация
- До 100 итераций оптимизации гиперпараметров
- Модель второго уровня обучается на предсказаниях моделей первого уровня
- `TabularUtilizedAutoML` запускает несколько конфигураций с разными `random_state` и объединяет результаты


In [4]:
lama_tuned = LAMABaseline(
    config_name="tuned",
    timeout=900,
    n_threads=4,
    n_folds=5,
    random_state=42,
    eda_insights=eda_insights,
)

lama_tuned.fit(train_df, target_col="loss", drop_cols=["id"])

metrics_tuned = lama_tuned.evaluate(y_true, return_all_metrics=True)

print(f"Log MAE: {metrics_tuned['log_mae']:.4f}")


2025-12-20 21:11:34,553 - src.models.baseline - INFO - Fitting LAMA baseline (tuned), shape: (188318, 132)
[21:11:34] Start automl utilizator with listed constraints:
[21:11:34] - time: 900.00 seconds
[21:11:34] - CPU: 4 cores
[21:11:34] - memory: 16 GB

[21:11:34] If one preset completes earlier, next preset configuration will be started

[21:11:34] ==================================================
[21:11:34] Start 0 automl preset configuration:
[21:11:34] conf_0_sel_type_0.yml, random state: {'reader_params': {'random_state': 42}, 'nn_params': {'random_state': 42}, 'general_params': {'return_all_predictions': False}}
[21:11:34] Stdout logging level is INFO.
[21:11:34] Task: reg

[21:11:34] Start automl preset with listed constraints:
[21:11:34] - time: 900.00 seconds
[21:11:34] - CPU: 4 cores
[21:11:34] - memory: 16 GB

[21:11:34] Train data shape: (188318, 132)

[21:11:40] Layer 1 train process start. Time left 894.19 secs
[21:11:44] Start fitting Lvl_0_Pipe_0_Mod_0_LinearL2 ...
[2

Optimization Progress:   0%|          | 0/100 [00:00<?, ?it/s]

2025-12-20 21:13:03,111 - optuna.storages._in_memory - INFO - A new study created in memory with name: no-name-b5cab15a-c938-4445-b9a1-a46dd4ec930c
2025-12-20 21:13:13,272 - optuna.study.study - INFO - Trial 0 finished with value: -1187.48341658606 and parameters: {'feature_fraction': 0.6872700594236812, 'num_leaves': 244, 'bagging_fraction': 0.8659969709057025, 'min_sum_hessian_in_leaf': 0.24810409748678114}. Best is trial 0 with value: -1187.48341658606.


Optimization Progress:   1%|          | 1/100 [00:10<16:46, 10.17s/it, best_trial=0, best_value=-1.19e+3]

2025-12-20 21:13:18,034 - optuna.study.study - INFO - Trial 1 finished with value: -1191.0728282674693 and parameters: {'feature_fraction': 0.5780093202212182, 'num_leaves': 53, 'bagging_fraction': 0.5290418060840998, 'min_sum_hessian_in_leaf': 2.915443189153755}. Best is trial 0 with value: -1187.48341658606.


...

Optimization Progress:  11%|█         | 11/100 [01:27<12:56,  8.73s/it, best_trial=4, best_value=-1.19e+3]

2025-12-20 21:24:49,957 - optuna.study.study - INFO - Trial 11 finished with value: -1188.6421801264853 and parameters: {'feature_fraction': 0.6075747423853284, 'num_leaves': 125, 'bagging_fraction': 0.871545830150035, 'min_sum_hessian_in_leaf': 0.02616865856724872}. Best is trial 11 with value: -1188.6421801264853.


Optimization Progress:  12%|█▏        | 12/100 [01:40<12:14,  8.35s/it, best_trial=11, best_value=-1.19e+3]

[21:24:49] Hyperparameters optimization for Lvl_1_Pipe_0_Mod_0_Tuned_LightGBM completed
[21:24:49] Start fitting Lvl_1_Pipe_0_Mod_0_Tuned_LightGBM ...


[21:25:26] Fitting Lvl_1_Pipe_0_Mod_0_Tuned_LightGBM finished. score = -1193.86867307248
[21:25:26] Lvl_1_Pipe_0_Mod_0_Tuned_LightGBM fitting and predicting completed
[21:25:26] Time left 68.21 secs

[21:25:26] Layer 2 training completed.

[21:25:26] Automl preset training completed in 204.52 seconds

[21:25:26] Model description:
Models on level 0:
	 5 averaged models Lvl_0_Pipe_0_Mod_0_LinearL2
	 5 averaged models Lvl_0_Pipe_1_Mod_0_LightGBM

Final prediction for new objects (level 1) = 
	 1.00000 * (5 averaged models Lvl_1_Pipe_0_Mod_0_Tuned_LightGBM) 

[21:25:26] ==================================================
[21:25:26] Blending: optimization starts with equal weights. Score = -1282.7782150
[21:25:26] Blending: iteration 0: score = -1186.8110449, weights = [0.4724345 0.        0.        0.        0.5275655]
[21:25:26] Blending: no improvements for score. Terminated.

[21:25:26] Blending: best score = -1186.8110449, best weights = [0.4724345 0.        0.        0.        0.52756

## Сравнение результатов
Сравним метрики качества двух конфигураций LAMA на кросс-валидации.


In [5]:
results = pd.DataFrame({
    'Configuration': ['Default', 'Tuned'],
    'Log MAE': [metrics_default['log_mae'], metrics_tuned['log_mae']],
})

print(results.to_string(index=False))

best_idx = results['Log MAE'].idxmin()
best_config = results.loc[best_idx, 'Configuration']
best_log_mae = results.loc[best_idx, 'Log MAE']

print(f"Best LAMA Configuration: {best_config}")

Configuration  Log MAE
      Default 0.523108
        Tuned 0.436323
Best LAMA Configuration: Tuned


## Генерация предсказаний на тестовой выборке

Используем лучшую модель для генерации предсказаний на тестовых данных


In [6]:
best_model = lama_default if metrics_default['log_mae'] < metrics_tuned['log_mae'] else lama_tuned

test_predictions = best_model.predict(test_df)

print(f"Predictions statistics:")
print(f"Mean: {test_predictions.mean():.2f}")
print(f"Std: {test_predictions.std():.2f}")
print(f"Min: {test_predictions.min():.2f}")
print(f"Max: {test_predictions.max():.2f}")

baseline_log_mae = best_log_mae
print(f"LAMA Baseline Log MAE to beat: {baseline_log_mae:.4f}")


Predictions statistics:
Mean: 3032.10
Std: 2172.39
Min: 585.96
Max: 39201.54
LAMA Baseline Log MAE to beat: 0.4363


## Выводы и анализ результатов

### Результаты экспериментов

| Конфигурация | Log MAE | Raw MAE | Улучшение |
|--------------|---------|---------|-----------|
| Default      | 0.5231  | ~1297   | —         |
| Tuned        | 0.4363  | ~1187   | **16.6%** |

### Ключевые наблюдения

1. В базовой конфигурации Linear L2 получила вес ~93.5% в финальном блендинге - это говорит о том, что линейная модель хорошо справляется с данной задаче. LightGBM добавляет лишь небольшое улучшение на первом уровне
2. Запуск нескольких конфигураций с разными `random_state` обеспечивает более робастные результаты

### Следующие шаги: Custom Solution (`03_custom_solution.ipynb`)

В следующем ноутбуке реализуем кастомное решение для сравнения с LAMA baseline:

1. **Три независимых пайплайна**: LightGBM, XGBoost и CatBoost с логарифмированием таргета
2. **Optuna-оптимизация**: автоматический подбор гиперпараметров LightGBM с оптимизацией Log MAE
3. **Взвешенный ансамбль**: объединение предсказаний моделей с весами, обратно пропорциональными их ошибкам
4. **EDA-инсайты**: использование рекомендаций из `01_eda.ipynb` (логарифмирование таргета, топовые MI-фичи)
